# Running Evaluation Benchmarks

### We will be running an evaluation benchmark provided with CHAMMI-75 dataset as an example. The benchmark we will be running is the RBC-MC benchmark, which evaluates the performance of a multi-class classification model on red blood cells morphology.

Library Imports

In [10]:
import pandas as pandas
import boto3
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import os
import sys
sys.path.append("../benchmarks/rbc-mc")

Downloading the RBC-MC Benchmark from AWS S3 Bucket

In [7]:
!mkdir -p rbc-mc

In [9]:
s3 = boto3.client('s3')

def download_file(args):
    bucket, key, local_path = args
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    s3.download_file(bucket, key, local_path)

# List all objects first
paginator = s3.get_paginator('list_objects_v2')
files = []
for page in paginator.paginate(Bucket='chammi-data', Prefix='CHAMMI-75/CHAMMI-75_test/rbc-mc/'):
    for obj in page.get('Contents', []):
        key = obj['Key']
        local_path = key.replace('CHAMMI-75/CHAMMI-75_test/rbc-mc/', 'rbc-mc/')
        files.append(('chammi-data', key, local_path))

# Download with progress bar
with ThreadPoolExecutor(max_workers=64) as executor:
    list(tqdm(executor.map(download_file, files), total=len(files)))

100%|██████████| 130008/130008 [09:23<00:00, 230.75it/s]


After downloading the dataset, we will extract features and then run the benchmark evaluation script.

In [ ]:
!accelerate launch --num_processes=1 extraction.py --model vit ../benchmarks/rbc-mc/extract_features.py --data_dir rbc-mc --output_dir rbc-mc/features